# Frozen vs blocks 3+4, on the cleaned dataset (2026-08-17)

The dataset changed enough that the last answer may not hold. Since the armA
build we removed 16,826 BirdNET clips (our own recordings, bird-labelled by a
machine, never listened to), 100 clips the species expert pulled out of
Background after listening, and 9 field *Colobus* clips that were being trained
on while also serving as the paper's positive control. 37,893 rows -> 21,120,
and Background falls from 68 % of the set to 43 %.

On the previous dataset unfreezing bought +0.002 and +0.001 at the two stations
where the metric is stable, and moved IPA4ST from 0.077 to 0.608 -- inside the
frozen run's own 0.077-0.962 spread on identical data, so unattributable. The
conclusion was that unfreezing is not the method.

Class balance has now shifted substantially, and how much of the trunk should
move is exactly the kind of thing that can depend on it. So the comparison is
run again: same folds, same patience, same data, only `--unfreeze` differs.

Read IPA20ST and IPA13ST. IPA4ST's evaluation pool is 100 calls in 2,470
detections, a 4 % base rate where precision is a knife edge and three runs of
one specification have given 0.962, 0.924 and 0.077.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!rm -rf /content/repo && git clone -q -b v13-honest-labels https://github.com/Mo119m/primates-sound-detection /content/repo
%cd /content/repo
!git log --oneline -1
U = '/content/drive/MyDrive/primates-sound-detection/upload_to_drive_CLEAN'
!mkdir -p /content/dataC && cp -n {U}/v13_images.npy {U}/v13_index.csv {U}/manifest.csv /content/dataC/
!ls -la /content/dataC


In [ ]:
A = '--manifest /content/dataC/manifest.csv --index /content/dataC/v13_index.csv --images /content/dataC/v13_images.npy --cache /content/dataC/v13_features.npy'
F = 'IPA20ST,IPA13ST,IPA4ST'

# The trainer refuses to train without a validated cache even on the
# fine-tuning path, which cost a whole run once. Build it first.
!python scripts/train_v13_loso.py --prepare-cache-only --overwrite {A} --out /content/out_clean_frozen.csv --run-metadata /content/cacheC.run.json

!python scripts/train_v13_loso.py --folds {F} --epochs 15 --patience 3 --overwrite {A} --out /content/out_clean_frozen.csv --head-dir /content/heads_clean_frozen --run-metadata /content/out_clean_frozen.run.json

!python scripts/train_v13_loso.py --folds {F} --epochs 15 --patience 3 --unfreeze 2 --finetune-epochs 5 --finetune-lr 1e-5 --overwrite {A} --out /content/out_clean_b34.csv --head-dir /content/heads_clean_b34 --run-metadata /content/out_clean_b34.run.json


In [ ]:
import os, shutil, pandas as pd
OUT = '/content/drive/MyDrive/primates-sound-detection/colab_results_clean'
os.makedirs(OUT, exist_ok=True)
for s in ['/content/heads_clean_frozen', '/content/heads_clean_b34',
          '/content/out_clean_frozen.csv', '/content/out_clean_b34.csv',
          '/content/out_clean_frozen.run.json', '/content/out_clean_b34.run.json']:
    if not os.path.exists(s):
        print(' ! missing, not copied:', s); continue
    d = os.path.join(OUT, os.path.basename(s))
    if os.path.isdir(s):
        shutil.rmtree(d, ignore_errors=True); shutil.copytree(s, d)
    else:
        shutil.copy(s, d)
    print(' ', os.path.basename(s))

fz = pd.read_csv('/content/out_clean_frozen.csv').set_index('station')
b34 = pd.read_csv('/content/out_clean_b34.csv').set_index('station')
print()
for st in ['IPA20ST', 'IPA13ST', 'IPA4ST']:
    a, b = fz.loc[st, 'loso_precision'], b34.loc[st, 'loso_precision']
    print(f'{st:9s} frozen {a:.4f}   blocks3+4 {b:.4f}   {b - a:+.4f}')
print()
print('Read the first two. IPA4ST is a 4 % base rate and does not hold still.')
